## Modeling Example Notebook for DTSC 4302

This notebook serves as a reference for how to use the personally defined APIs in order to run and interpret Mixed Effects Models in this project.

In [2]:
# non built-in libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import statsmodels.api as sm
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# built-in libraries
import os
import json
from pathlib import Path
import sys
import datetime
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# personally-defined modules
sys.path.append(os.path.join(str(Path.cwd()), "../"))  
from scripts.data_download import download_files_from_fld
from scripts.modeling.mixed_model_wrapper import fit_mixed_model, plot_diagnostics, anova_mixed_models

# install reqd. datasets from Google Drive
folder_id = "1P2FRAkPrqL2nn2MNMyd4ilWbXNS_kkKD" 
data_path = os.path.join(str(Path.cwd()), "../data")
download_files_from_fld(folder_id, data_path)

# constants
START_DATE = datetime.datetime(2013, 10, 1)  # init start date of analysis to first day of FY 2014
END_DATE = datetime.datetime(2024, 9, 30)  # init end date of analysis to last day of FY 2024

# reading in data
sent_df = pd.read_csv(os.path.join(data_path, "sentencing_data_cleaned.csv"), low_memory=False)
districts_gdf = gpd.read_file(os.path.join(data_path, "us_district_cts_bounds.geojson"))
districts_gdf = districts_gdf.rename(columns={c: c.upper() for c in districts_gdf.columns})
districts_gdf = districts_gdf[["NAME", "DISTRICT_N", "GEOMETRY"]]

All files already downloaded.


## Example Model Run # 1

This is an example showing a simple model fit on the sentencing data with random intercepts for each district and a global intercept. This example also shows how to interpret the results of the model.

In [ ]:
# filter to only include people who received atleast one month of prison
expmnt_df = sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 1].copy()

# scale MNTHS_PRSN_NO_ALT to reduce runtime of mixed model fitting and help convergence
expmnt_df["MNTHS_PRSN_NO_ALT"] = expmnt_df["MNTHS_PRSN_NO_ALT"] / 12  

expmnt_df["MNTHS_PRSN_NO_ALT"] = np.log(expmnt_df["MNTHS_PRSN_NO_ALT"])  # log transform to reduce skew

result = fit_mixed_model(
    expmnt_df[["MNTHS_PRSN_NO_ALT", "DIST_CRT"]],   # only inlcude columns used in formula to save time
    formula="MNTHS_PRSN_NO_ALT ~ (1 | DIST_CRT)",  # use "||" instead of "|" to enforce diagonal covariance structure
    family="gaussian",  # binomial if outcome is binary
    link=None,  # must be None for family="gaussian", see docstring for more details
    categorical_cols=["DIST_CRT"],  # all columns that you want to be treated as categorical
    reference_levels = None,
    return_confint=False,  # return confidence intervals for fixed effects
    return_random_effects_variance=True,
    return_random_effects_covariance=True,  # return covariance matrices for random effects.
    return_fitted=True,  # return fitted values (reqd for calling plot_diagnostics function). Set to False if you want to save time and dont care about diagnostics
    return_residuals=True,  # return pearson studentized residuals (reqd for calling plot_diagnostics function). Set to False if you want to save time and dont care about diagnostics
    optimizer = "bobyqa",  # experiment with changing optimizer if you receive convergence issuses
    keep_raw_summary=True,  # leave True to see model summary produced by lmer4
    return_random_effects=True,  # return estimated random effect magintudes (there is no p-values or std errors)
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  # dont change this
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe"  # absolute path to your Rscript.exe file
)
print(result.raw_summary)
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Fit Statistics

- Higher AIC and higher BIC are better
- lower 'logLik' (log likelihood) is better
- 'nobs' is the number of data points used to fit the model
- 'df_residual' is the degreees of freedom of residuals
- `sigma` is the standard deviation of residuals
- `REMLCrit` is the value of the restricted maximum likelihood at the end of fitting  

In [ ]:
result.fit_statistics

### Estimated Standard Deviation of Random Effects (and residuals) 

Recall that random effects for all *j* groups are assumed to be i.i.d from multivariate normal distrbution with mean 0  and covariance matrix G. In other words $u_j \sim N_{p+1}(0, G)$. So the estimates in the table below are the estimates of the square root of the diagonal elements of the covariance matrix *G*.

In [ ]:
result.random_effects_variance

When fitting the random intercepts only model, the Intraclass Correlation Coefficient (ICC) can be calculated as

$$ICC = \frac{\sigma_{group}^2}{\sigma_{group}^2 + \sigma_{residual}^2}$$

In [ ]:
sigma_group = result.random_effects_variance.iloc[0, 3]**2
sigma_residual = result.random_effects_variance.iloc[1, 3]**2

icc = sigma_group / (sigma_group + sigma_residual)
print(f"Intraclass Correlation Coefficient (ICC): {icc:.4f}")

### Random Effects Covariance Matrix

In [ ]:
result.random_effects_covariance_matrices["DIST_CRT"]

### Estimates of Random Effects for Each District 

In [ ]:
result.random_effects.head(2)

We can use a choropleth map to better analyze these estimates

In [ ]:
dist_estimates = result.random_effects.merge(
    districts_gdf,
    left_on="level",
    right_on="NAME",
    how="left"
)

dist_estimates_gdf = gpd.GeoDataFrame(dist_estimates, geometry="GEOMETRY")

# lon/lat coords
dist_estimates_gdf = dist_estimates_gdf.to_crs(epsg=4326)

dist_estimates_gdf = dist_estimates_gdf.reset_index(drop=True)
dist_estimates_gdf["district_id"] = dist_estimates_gdf.index.astype(str)

geojson = json.loads(dist_estimates_gdf.to_json())

fig = px.choropleth(
    dist_estimates_gdf,
    geojson=geojson,
    locations="district_id",
    featureidkey="properties.district_id",
    color="estimate",
    color_continuous_scale="RdBu",
    title="Random Effect Estimates by District",
    labels={"estimate": "Random Effect Estimate"},
    hover_name="level",
    hover_data={"estimate": True, "district_id": False}
)

# center on nebraska
fig.update_geos(
    center={"lat": 41.5, "lon": -99.8},
    projection_scale=5, 
    visible=False
)
fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
fig.show()

### Visualize Residual Diagnostics

In [ ]:
plot_diagnostics(result)  # reqs that result was fitted with return_fitted=True and return_residuals=True

## Example Model Run # 2

This model run fits a model with a a fixed intercept and random slopes and intercepts for each district (see the argument for `formula`).

In [ ]:
# filter to only include people who received atleast one month of prison
expmnt_df = sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 1].copy()  

# scale MNTHS_PRSN_NO_ALT to reduce runtime of mixed model fitting and help convergence
expmnt_df["MNTHS_PRSN_NO_ALT"] = expmnt_df["MNTHS_PRSN_NO_ALT"] / 12  

expmnt_df["MNTHS_PRSN_NO_ALT"] = np.log(expmnt_df["MNTHS_PRSN_NO_ALT"])  # log transform to reduce skew

result = fit_mixed_model(
    expmnt_df[["MNTHS_PRSN_NO_ALT", "DIST_CRT", "RACE"]], 
    formula="MNTHS_PRSN_NO_ALT ~ (1 + RACE | DIST_CRT)",  
    family="gaussian",  
    link=None,  
    categorical_cols=["DIST_CRT", "RACE"], 
    reference_levels = {"RACE": "White"},
    return_confint=False,  
    return_random_effects_variance=True,
    return_random_effects_covariance=True,  
    return_fitted=True, 
    return_residuals=True,  
    optimizer = "bobyqa",  
    keep_raw_summary=True, 
    return_random_effects=True,  
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe" 
)
print(result.raw_summary)
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Interpreting Results

The correlation between the random intercept and random slopes are relatively low, indicating the model is not ill-conditioned. Additionally, the fixed intercept appears to be extremely significant.



### Ensure there are no Convergence Issues

Luckily, there are no convergence messages with this model run, so we can continue to interpret results.

In [ ]:
result.diagnostics["convergence_messages"]

### Fit Statistics

In [ ]:
result.fit_statistics

### Random Effects Covariance/Correlation Matrix

Now that we have multiple random effects, we will use a correlation matrix instead of a covariance matrix to allow for easier comparison of relatiopnships. Since we passed `reference_levels = {"RACE": "White"}` when fitting the model, the interpretation of the RACE slopes is that it is the expected difference in the response variable from White offenders.

In [ ]:
# convert covariance matrix into correlation matrix for easier interpretation of relationships between random effects
cov_matrix = result.random_effects_covariance_matrices["DIST_CRT"]
diag = np.sqrt(np.diag(cov_matrix))
cor_matrix = cov_matrix / np.outer(diag, diag)
cor_matrix

Based on the correlation matrix, it appears that districts with higher baseline sentencing (for White offenders) tend to exhibit less additional harshness toward Black offenders relative to Whites (r=-0.55). Additionally, it appers that districts with higher gaps in harshness between hispanic and white offenders also tend to have higher gaps in harshness between black and white offenders.

### Random Effects Estimates for Each District

In [ ]:
result.random_effects.head(2)

Choropleth map to investgate random slopes for RACEBlack across districts

In [ ]:
dist_estimates = (
    result.random_effects[result.random_effects["term"]=="RACEBlack"]  # filter to only include random effect estimates for Black vs White disparity
    .merge(districts_gdf, left_on="level", right_on="NAME",how="left"
))

dist_estimates_gdf = gpd.GeoDataFrame(dist_estimates, geometry="GEOMETRY")

# lon/lat coords
dist_estimates_gdf = dist_estimates_gdf.to_crs(epsg=4326)

dist_estimates_gdf = dist_estimates_gdf.reset_index(drop=True)
dist_estimates_gdf["district_id"] = dist_estimates_gdf.index.astype(str)

geojson = json.loads(dist_estimates_gdf.to_json())

fig = px.choropleth(
    dist_estimates_gdf,
    geojson=geojson,
    locations="district_id",
    featureidkey="properties.district_id",
    color="estimate",
    color_continuous_scale="RdBu",
    title="Random Effect Estimates by District",
    labels={"estimate": "Random Effect Estimate"},
    hover_name="level",
    hover_data={"estimate": True, "district_id": False}
)

# center on nebraska
fig.update_geos(
    center={"lat": 41.5, "lon": -99.8},
    projection_scale=5, 
    visible=False
)
fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
fig.show()

### Visualize Residual Diagnostics

In [ ]:
plot_diagnostics(result) 

## Example Model Run 3

This model is going to be the same as Model 2 but with an additional fixed effect for offense level. Additionaly, this example shows how to test which model is better using an ANOVA test. 

Note, to test which model is better using the ANOVA test, the null model should be a nested version of the alternative model. Specifically, **the nested model should have the same random effects but a subset of the fixed effects**. 

In [ ]:
# filter to only include people who received atleast one month of prison
expmnt_df = sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 1].copy()  

# scale MNTHS_PRSN_NO_ALT to reduce runtime of mixed model fitting and help convergence
expmnt_df["MNTHS_PRSN_NO_ALT"] = expmnt_df["MNTHS_PRSN_NO_ALT"] / 12  

expmnt_df["MNTHS_PRSN_NO_ALT"] = np.log(expmnt_df["MNTHS_PRSN_NO_ALT"])  # log transform to reduce skew

result = fit_mixed_model(
    expmnt_df[["MNTHS_PRSN_NO_ALT", "DIST_CRT", "RACE", "OFF_TYPE"]], 
    formula="MNTHS_PRSN_NO_ALT ~ OFF_TYPE + (1 + RACE | DIST_CRT)",  
    family="gaussian",  
    link=None,  
    categorical_cols=["DIST_CRT", "RACE", "OFF_TYPE"], 
    reference_levels = {"RACE": "White", "OFF_TYPE": "Drugs"},
    return_confint=False,  
    return_random_effects_variance=True,
    return_random_effects_covariance=True,  
    return_fitted=True, 
    return_residuals=True,  
    optimizer = "bobyqa",  
    keep_raw_summary=True, 
    return_random_effects=True,  
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe" 
)
print(result.raw_summary)
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Ensure there are no Convergence Issues

Luckily, there are no convergence messages with this model run, so we can continue to interpret results.

In [ ]:
result.diagnostics["convergence_messages"]

### Fit Statistics

In [ ]:
result.fit_statistics

### Random Effects Covariance/Correlation Matrix

Note that after including OFF_TYPE in the model, the correlations/covariances of the random effects changes. This also implies that the estimates for the  random effects also change.

In [ ]:
# convert covariance matrix into correlation matrix for easier interpretation of relationships between random effects
cov_matrix = result.random_effects_covariance_matrices["DIST_CRT"]
diag = np.sqrt(np.diag(cov_matrix))
cor_matrix = cov_matrix / np.outer(diag, diag)
cor_matrix

### Testing Which Model is Better

Clearly, since Model 3 has more parameters than Model 2, it will fit the data better and will have a higher likelihood value. However, we would like to test if this difference is justifiable for the decrease in degrees of freedom and increase in model complexity.

Note that the p-values are actually more conservative than they should be. This is not necesarily a bad thing because if we observe significance at alpha=0.05, then the real p-value would also certainly be significant.

Note that this function does take a very long time to run, so tests should be meaninfully thought out to avoid wasting time and potentially p-hacking.

In [ ]:
anova_result = anova_mixed_models(
    data=expmnt_df[["MNTHS_PRSN_NO_ALT", "DIST_CRT", "RACE", "OFF_TYPE"]].copy(),
    null_formula = "MNTHS_PRSN_NO_ALT ~ (1 + RACE | DIST_CRT)",
    alt_formula = "MNTHS_PRSN_NO_ALT ~ OFF_TYPE + (1 + RACE | DIST_CRT)",
    family = "gaussian",
    link = None,
    categorical_cols = ["RACE", "OFF_TYPE"],
    reference_levels = {"RACE": "White", "OFF_TYPE": "Drugs"},
    optimizer = "bobyqa",
    test_type = "ml",  # use "ml" for testing differences in fixed effects
    r_script_path = os.path.join("r", "fit_mixed_model.R"),
    r_executable = "c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe",
) 

In [ ]:
anova_result.comparison_table

The p-value, Pr(>Chisq), provides strong evidence that our alternative model is a better fit and it may be worthwile to include OFF_TYPE in the model.

## Example Model Run 4

In this model run, we will model incarceration instead of sentence lenght. Clearly, the `gaussian` family is no longer appropriate and we need to move to the `binomial` family to respect the boundaries of the dependent variable, which is binary.

This model will be similar to Model 3 in that we include fixed effects for offense type and random intercept and a random slope for the race of the offender.|

In [ ]:
expmnt_df = sent_df[sent_df["RCVD_PRIS_SENT_ELGB_PROBAT"].notna()].copy()

result = fit_mixed_model(
    expmnt_df[["RCVD_PRIS_SENT_ELGB_PROBAT", "DIST_CRT", "RACE", "SEX"]], 
    formula="RCVD_PRIS_SENT_ELGB_PROBAT ~ SEX + (1 + RACE | DIST_CRT)",  
    family="binomial",  
    link="logit",  # can also use "probit" or "cloglog" for binary outcomes, see docstring for more details 
    categorical_cols=["DIST_CRT", "RACE", "SEX"], 
    reference_levels = {"RACE": "White", "SEX": "Male"},
    return_confint=False,  
    return_random_effects_variance=True,
    return_random_effects_covariance=True,  
    return_fitted=True, 
    return_residuals=True,  
    optimizer = "bobyqa",  
    keep_raw_summary=True, 
    return_random_effects=True,  
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe" 
)
print(result.raw_summary)
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Ensure there are no Convergence Issues

Luckily, there are no convergence messages with this model run, so we can continue to interpret results.

In [ ]:
result.diagnostics["convergence_messages"]

### Fit Statistics

In [ ]:
result.fit_statistics

### Random Effects Covariance/Correlation Matrix



In [ ]:
# convert covariance matrix into correlation matrix for easier interpretation of relationships between random effects
cov_matrix = result.random_effects_covariance_matrices["DIST_CRT"]
diag = np.sqrt(np.diag(cov_matrix))
cor_matrix = cov_matrix / np.outer(diag, diag)
cor_matrix

### Random Effects Estimates for Each District

In [ ]:
result.random_effects.head(2)

Choropleth map to investgate random slopes for RACEBlack across districts

In [ ]:
dist_estimates = (
    result.random_effects[result.random_effects["term"]=="RACEBlack"]  # filter to only include random effect estimates for Black vs White disparity
    .merge(districts_gdf, left_on="level", right_on="NAME",how="left"
))

dist_estimates_gdf = gpd.GeoDataFrame(dist_estimates, geometry="GEOMETRY")

# lon/lat coords
dist_estimates_gdf = dist_estimates_gdf.to_crs(epsg=4326)

dist_estimates_gdf = dist_estimates_gdf.reset_index(drop=True)
dist_estimates_gdf["district_id"] = dist_estimates_gdf.index.astype(str)

geojson = json.loads(dist_estimates_gdf.to_json())

fig = px.choropleth(
    dist_estimates_gdf,
    geojson=geojson,
    locations="district_id",
    featureidkey="properties.district_id",
    color="estimate",
    color_continuous_scale="RdBu",
    title="Random Effect Estimates by District",
    labels={"estimate": "Random Effect Estimate"},
    hover_name="level",
    hover_data={"estimate": True, "district_id": False}
)

# center on nebraska
fig.update_geos(
    center={"lat": 41.5, "lon": -99.8},
    projection_scale=5, 
    visible=False
)
fig.update_layout(margin={"r": 0, "t": 30, "l": 0, "b": 0})
fig.show()

### Compute Accuracy

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# make classification report using fitted values from mixed model with binary outcome
y_true = expmnt_df["RCVD_PRIS_SENT_ELGB_PROBAT"]
y_pred_prob = result.diagnostics["fitted"]
y_pred = (y_pred_prob >= 0.5).astype(int)  # convert probabilities to binary predictions using 0.5 threshold
print(classification_report(y_true, y_pred))


## Austin - Modeling

In [ ]:
sent_df.columns[:100]

In [ ]:
sent_df['RACE'].value_counts()

In [ ]:
# filter to only include people who received atleast one month of prison
expmnt_df = sent_df[sent_df["MNTHS_PRSN_NO_ALT"] > 1].copy()

# Create dummy variables
dummies = pd.get_dummies(expmnt_df, columns=['CTRY_OF_CITZNSHIP'])
df = pd.concat([expmnt_df, dummies], axis=1)
# scale MNTHS_PRSN_NO_ALT to reduce runtime of mixed model fitting and help convergence
expmnt_df["MNTHS_PRSN_NO_ALT"] = expmnt_df["MNTHS_PRSN_NO_ALT"] / 12  

expmnt_df["MNTHS_PRSN_NO_ALT"] = np.log(expmnt_df["MNTHS_PRSN_NO_ALT"])  # log transform to reduce skew

result = fit_mixed_model(
    expmnt_df[["MNTHS_PRSN_NO_ALT", "DIST_CRT", "RACE", "OFF_TYPE", "CTRY_OF_CTZNSHP"]], 
    formula="MNTHS_PRSN_NO_ALT ~ OFF_TYPE + (1 + RACE | DIST_CRT)",  
    family="gaussian",  
    link=None,  
    categorical_cols=["DIST_CRT", "RACE", "OFF_TYPE", "CTRY_OF_CTZNSHP"], 
    reference_levels = {"RACE": "White", "OFF_TYPE": "Drugs", "CTRY_OF_CTZNSHP": "USA"},
    return_confint=False,  
    return_random_effects_variance=True,
    return_random_effects_covariance=True,  
    return_fitted=True, 
    return_residuals=True,  
    optimizer = "bobyqa",  
    keep_raw_summary=True, 
    return_random_effects=True,  
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe" 
)
print(result.raw_summary)
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

## New

In [3]:
sent_df = sent_df[
    (sent_df["DETENT_ST"] != "Other") &
    (sent_df["DETENT_ST"].notna()) &
    (sent_df["EDUCATION"].notna())
]

sent_df["GOVT_SPON_DEPT"] = (
    (sent_df["5K1.1"] == 1) | 
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(["Govt. Sponsored Departure", "Early Disposition/5K3.1"])) |
    (sent_df["SENT_RANGE_2004_2017"].isin(["Government Sponsored - Below Range"]))
).astype(int)

# dependent variable for one of the models (only defined for FY >= 2018)
sent_df["DOWNWARD_VAR"] = sent_df["SENT_RANGE_2018_PRSNT"].isin([
    "Govt. Sponsored Variance", "Below Range Variance"
]).astype(int)

# create guideline midpoint in years
sent_df["GL_MDPT"] = (sent_df["GL_MIN"] + sent_df["GL_MAX"]) / 24

# dependent variable for if someone was sentenced above/below the guideline midpoint
sent_df["SENT_ABOVE_GL_MDPT"] = ((sent_df["MNTHS_PRSN_NO_ALT"] / 12) >= sent_df["GL_MDPT"]).astype(int)

# treat CHC as categorical 
sent_df["CHC"] = sent_df["CHC"].astype(int).astype(str) 

# binary flags for whether the statutory minimum sentence is zero
sent_df["STAT_MIN_ZERO_FLAG"] = (sent_df["STAT_MIN"] == 0).astype(int)

# convert months of prison to years for easier model convergence
sent_df["YEARS_PRSN_NO_ALT"] = sent_df["MNTHS_PRSN_NO_ALT"] / 12

# binary flag for whether the defendant is a U.S. citizen
sent_df["CITIZEN"] = (sent_df["CITIZEN"] == "U.S. Citizen").astype(int)

# binary flag indicating whether the defendant commited the offense while under supervision for a prior offense
sent_df["OFF_CMTD_UNDER_SUP"] = (sent_df["OFF_CMTD_UNDER_SUP"] > 0).astype(int)

# easier convergence
sent_df["AGE"] /= 100
sent_df["OL"] /= 43

# make reference levels for categorical variables more explicit
ref_levels = {
    "CHC": "1",
    "EDUCATION": "Graduate Degree",
    "DETENT_ST": "In Custody",
    "OFF_TYPE": "Drugs",
    "SEX": "Male",
    "RACE": "White",
}

### Model 1

In [ ]:
expmnt_df = sent_df[
    (sent_df["EDUCATION"].notna()) &
    (sent_df["DETENT_ST"].notna())
].copy()


cols_to_incl = [
    "RECIEVED_PRSN_FLAG", "DIST_CRT", "RACE", "SEX", "OL", "CHC",
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG", "STAT_MIN_ZERO_FLAG",
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS",
    "PCT_REPUBLICAN", "PCT_MALE", "PCT_WHITE", "EDUCATION"
]


categorical_cols = ["DIST_CRT", "RACE", "SEX", "CHC", "OFF_TYPE", "DETENT_ST", "EDUCATION"]


formula = "RECIEVED_PRSN_FLAG ~ " + \
    "poly(OL, 4) + CHC + RACE + SEX + DETENT_ST + poly(AGE, 2) + STAT_MIN_ZERO_FLAG " +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + PCT_REPUBLICAN + PCT_MALE + PCT_WHITE + EDUCATION " +\
    "+ (1 + RACE + SEX | DIST_CRT)"


result = fit_mixed_model(
    expmnt_df[cols_to_incl],
    formula=formula,  
    family="binomial",  
    link="logit",
    categorical_cols= categorical_cols,
    reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols},  
    optimizer = "bobyqa",  
    n_iter = 20_000,
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe",
    save_result_path=os.path.join("models", "prison_sentence_model_all_years.json")
)
print(result.raw_summary)
print(result.diagnostics["convergence_messages"])
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")

### Model 2

In [4]:
expmnt_df = sent_df[
    (sent_df["FISCAL_YR"] >= 2018) |
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(
        ["Within Range", "Below Range Variance", "Govt. Sponsored Variance", "Above Range Variance"]
    ))
].copy()


cols_to_incl = [
    "DOWNWARD_VAR", "DIST_CRT", "RACE",
    "EDUCATION", "OFF_TYPE", "SEX", "AGE",
    "ANY_CRIM_HIST_FLAG", "DETENT_ST",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS",
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE"
]


categorical_cols = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]


result = fit_mixed_model(
    expmnt_df[cols_to_incl],  
    formula="DOWNWARD_VAR ~ " \
    "RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG "
    "+ log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) "
    "+ OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN "
    "+ PCT_REPUBLICAN + PCT_MALE "
    "+ (1 + RACE + SEX | DIST_CRT)",  
    family="binomial",  
    categorical_cols=categorical_cols,
    reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols},
    optimizer = "bobyqa",  
    n_iter=20_000,
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe",
    save_result_path=os.path.join("models", "no_nulls_variance_or_not_no_departures_all_years.json")
)
print(result.diagnostics["convergence_messages"])
print(result.raw_summary)

Fitting mixed model with formula: DOWNWARD_VAR ~ RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG + log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) + OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN + PCT_REPUBLICAN + PCT_MALE + (1 + RACE + SEX | DIST_CRT), family: binomial, optimizer: bobyqa...
Seconds taken to fit mixed model: 20620.37
[]
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: DOWNWARD_VAR ~ RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG +  
    log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) + OFF_TYPE +  
    poly(AGE, 2) + EDUCATION + CITIZEN + PCT_REPUBLICAN + PCT_MALE +  
    (1 + RACE + SEX | DIST_CRT)
   Data: data
 Offset: if (!is.null(offset)) data[[offset]] else NULL
Control: ctrl

      AIC       BIC    logLik -2*log(L)  df.resid 
 316533.5  317037.2 -158218.8  316437.5    266648 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.1449 -0.7348 -0.4789  0.9442 15.6906 

Ra

### Model 2  - no random race effect

In [5]:
expmnt_df = sent_df[
    (sent_df["FISCAL_YR"] >= 2018) |
    (sent_df["SENT_RANGE_2018_PRSNT"].isin(
        ["Within Range", "Below Range Variance", "Govt. Sponsored Variance", "Above Range Variance"]
    ))
].copy()


cols_to_incl = [
    "DOWNWARD_VAR", "DIST_CRT", "RACE",
    "EDUCATION", "OFF_TYPE", "SEX", "AGE",
    "ANY_CRIM_HIST_FLAG", "DETENT_ST",
    "TRIAL_FLAG", "CHC", "OL", "NUM_COUNTS",
    "NUM_DEPENDENTS", "CITIZEN", "PCT_REPUBLICAN",
    "PCT_MALE"
]


categorical_cols = [
    "DIST_CRT", "RACE", "EDUCATION", "OFF_TYPE", "SEX", "DETENT_ST", "CHC"
]


result = fit_mixed_model(
    expmnt_df[cols_to_incl],  
    formula="DOWNWARD_VAR ~ " \
    "RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG "
    "+ log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) "
    "+ OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN "
    "+ PCT_REPUBLICAN + PCT_MALE "
    "+ (1 + SEX | DIST_CRT)",  
    family="binomial",  
    categorical_cols=categorical_cols,
    reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols},
    optimizer = "bobyqa",  
    n_iter=20_000,
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe",
    save_result_path=os.path.join("models", "no_rand_race_no_nulls_variance_or_not_no_departures_all_years.json")
)
print(result.diagnostics["convergence_messages"])
print(result.raw_summary)

Fitting mixed model with formula: DOWNWARD_VAR ~ RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG + log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) + OFF_TYPE + poly(AGE, 2) + EDUCATION + CITIZEN + PCT_REPUBLICAN + PCT_MALE + (1 + SEX | DIST_CRT), family: binomial, optimizer: bobyqa...
Seconds taken to fit mixed model: 785.23
[]
Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: DOWNWARD_VAR ~ RACE + SEX + poly(OL, 4) + CHC + TRIAL_FLAG +  
    log1p(NUM_DEPENDENTS) + DETENT_ST + log(NUM_COUNTS) + OFF_TYPE +  
    poly(AGE, 2) + EDUCATION + CITIZEN + PCT_REPUBLICAN + PCT_MALE +  
    (1 + SEX | DIST_CRT)
   Data: data
 Offset: if (!is.null(offset)) data[[offset]] else NULL
Control: ctrl

      AIC       BIC    logLik -2*log(L)  df.resid 
 317301.9  317679.6 -158614.9  317229.9    266660 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.2054 -0.7362 -0.4847  0.9499 16.8979 

Random effects:
 G

### Model 3

In [ ]:
expmnt_df = sent_df[
    (sent_df["EDUCATION"].notna()) &
    (sent_df["DETENT_ST"].notna()) &
    (sent_df["MNTHS_PRSN_NO_ALT"] > 0.03)  # consider filtering > 1
].copy()


expmnt_df["LOG_SENTENCE_LENGTH_YEARS"] = np.log(expmnt_df["YEARS_PRSN_NO_ALT"])
expmnt_df["LOG_GL_MDPT"] = np.log(expmnt_df["GL_MDPT"])


cols_to_incl = [
    "LOG_SENTENCE_LENGTH_YEARS", "DIST_CRT", "RACE", "SEX", "OL", "CHC",
    "DETENT_ST", "AGE", "OFF_TYPE", "TRIAL_FLAG",
    "GOVT_SPON_DEPT", "CITIZEN", "NUM_DEPENDENTS", "NUM_COUNTS",
    "PCT_REPUBLICAN", "PCT_MALE", "PCT_WHITE", "EDUCATION", "LOG_GL_MDPT"
]


categorical_cols = ["DIST_CRT", "RACE", "SEX", "CHC", "OFF_TYPE", "DETENT_ST", "EDUCATION"]


formula = "LOG_SENTENCE_LENGTH_YEARS ~ " + \
    "poly(OL, 4) + CHC + RACE + SEX + DETENT_ST + poly(AGE, 2)" +\
    "+ OFF_TYPE + TRIAL_FLAG + GOVT_SPON_DEPT + CITIZEN + log1p(NUM_DEPENDENTS) " +\
    "+ log(NUM_COUNTS) + PCT_REPUBLICAN + PCT_MALE  + EDUCATION + LOG_GL_MDPT " +\
    "+ (1 + RACE + SEX | DIST_CRT)"


result = fit_mixed_model(
    expmnt_df[cols_to_incl],
    formula=formula,  
    family="gaussian",  
    categorical_cols= categorical_cols,
    reference_levels = {k: v for k, v in ref_levels.items() if k in categorical_cols},  
    optimizer = "bobyqa",  
    n_iter = 20_000,  
    r_script_path=os.path.join("r", "fit_mixed_model.R"),  
    r_executable="c:\\Program Files\\R\\R-4.5.2\\bin\\Rscript.exe",
    save_result_path=os.path.join("models", "log_sentence_length_all_years_with_gl_mdpt_and_sents_LT_1_month.json")
)
print(result.raw_summary)
print(result.diagnostics["convergence_messages"])
print(f"\n\nErrors:\n", result.errors or "None", "\n\nWarnings:\n", result.warnings or "None")